# 07 — Discount Impact on Profitability
How discounting erodes profit margins — bucketed analysis and correlation.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Profit by Discount Bucket ─────────────────────────────────────────────────
buckets = fetch_df(cur, """
    SELECT
        CASE
            WHEN discount = 0    THEN '0% (No Discount)'
            WHEN discount <= 0.1 THEN '1-10%'
            WHEN discount <= 0.2 THEN '11-20%'
            WHEN discount <= 0.3 THEN '21-30%'
            WHEN discount <= 0.5 THEN '31-50%'
            ELSE '>50%'
        END AS discount_bucket,
        COUNT(*)                       AS line_items,
        ROUND(SUM(sales),2)            AS revenue,
        ROUND(SUM(profit),2)           AS profit,
        ROUND(AVG(profit/sales)*100,2) AS avg_margin_pct
    FROM order_items
    GROUP BY
        CASE
            WHEN discount = 0    THEN '0% (No Discount)'
            WHEN discount <= 0.1 THEN '1-10%'
            WHEN discount <= 0.2 THEN '11-20%'
            WHEN discount <= 0.3 THEN '21-30%'
            WHEN discount <= 0.5 THEN '31-50%'
            ELSE '>50%'
        END
    ORDER BY avg_margin_pct DESC
""")

profit_colors = [PALETTE[1] if v >= 0 else PALETTE[3]
                 for v in buckets["profit"]]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(buckets["discount_bucket"], buckets["profit"],
              color=profit_colors, edgecolor="white", zorder=3)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")

for bar, mg in zip(bars, buckets["avg_margin_pct"]):
    h = bar.get_height()
    offset = 500 if h >= 0 else -1500
    ax.text(bar.get_x() + bar.get_width()/2,
            h + offset,
            f"{fmt_usd(h)}\n({mg}% margin)",
            ha="center", va="bottom", fontsize=8)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.set_title("Total Profit by Discount Bucket")
ax.set_ylabel("Profit (USD)")
ax.set_xlabel("Discount Range")
plt.xticks(rotation=20)
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Scatter: Discount % vs Profit per Line Item ───────────────────────────────
raw = fetch_df(cur, """
    SELECT ROUND(discount*100, 0) AS discount_pct,
           ROUND(profit, 2)       AS profit
    FROM order_items LIMIT 2000
""")

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(raw["discount_pct"], raw["profit"],
           alpha=0.35, s=12, color=PALETTE[0], edgecolors="none")
ax.axhline(0, color=PALETTE[3], linewidth=1.2, linestyle="--",
           label="Break-even")

# Compute and plot simple linear trend
import numpy as np
m, b = np.polyfit(raw["discount_pct"], raw["profit"], 1)
xs = sorted(raw["discount_pct"].unique())
ax.plot(xs, [m*x + b for x in xs], color=PALETTE[2],
        linewidth=2, linestyle="-", label=f"Trend  (slope={m:.1f})")

ax.set_title("Discount % vs Profit per Line Item")
ax.set_xlabel("Discount (%)")
ax.set_ylabel("Profit (USD)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
